# Subfase 8: Evaluación comparativa (manifests)

Objetivo: consolidar métricas de modelos clásicos y cuánticos a partir de los manifests JSON

Salida: `results/fase4_comparativa/BVG_subfase8_comparativa_h5.csv`

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

ROOT = Path('..')
CLASSICAL_DIR = ROOT / 'models' / 'classical'
QUANTUM_DIR = ROOT / 'models' / 'quantum'
OUT_DIR = ROOT / 'results' / 'fase4_comparativa'
OUT_FILE = OUT_DIR / 'BVG_subfase8_comparativa_h5.csv'

MANIFEST_FIELDS = ['company', 'model_family', 'metrics']
METRICS_FIELDS = ['accuracy', 'f1']

def _assert_manifest_fields(payload: dict, path: Path) -> None:
    missing = [field for field in MANIFEST_FIELDS if field not in payload]
    if missing:
        raise ValueError(f'Manifest incompleto: {path.name}. Faltan campos: {missing}')
    if not isinstance(payload.get('metrics'), dict):
        raise ValueError(f'Manifest {path.name} tiene metrics inválido (se esperaba dict).')

def _read_manifest(path: Path) -> dict:
    with path.open('r', encoding='utf-8') as handle:
        payload = json.load(handle)
    _assert_manifest_fields(payload, path)
    payload['__manifest_path__'] = path
    return payload

def load_manifests(paths: list[Path]) -> list[dict]:
    manifests = [_read_manifest(path) for path in paths]
    if not manifests:
        raise ValueError('No se encontraron manifests para procesar.')
    return manifests

def build_comparison_df(manifests: list[dict]) -> pd.DataFrame:
    rows = []
    for payload in manifests:
        metrics = payload.get('metrics', {})
        model_name = payload.get('model_name')
        if not model_name:
            model_name = payload['__manifest_path__'].stem.replace('_manifest', '')
        horizonte = payload.get('horizon') or payload.get('horizonte', 'h5')
        row = {
            'company': payload.get('company'),
            'model_family': payload.get('model_family'),
            'model_name': model_name,
            'horizonte': horizonte,
            'accuracy': metrics.get('accuracy'),
            'f1': metrics.get('f1'),
        }
        for optional_key in ['train_end_date', 'created_at_utc']:
            if optional_key in payload:
                row[optional_key] = payload.get(optional_key)
        rows.append(row)

    df = pd.DataFrame(rows)
    required_cols = ['company', 'model_family', 'model_name', 'horizonte', 'accuracy', 'f1']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f'Columnas requeridas ausentes en DataFrame: {missing_cols}')

    df = df.sort_values(['company', 'model_family', 'model_name']).reset_index(drop=True).copy()
    return df

def export_comparison(df: pd.DataFrame, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)

classical_manifests = sorted(CLASSICAL_DIR.glob('*_manifest.json'))
quantum_manifests = sorted(QUANTUM_DIR.glob('*_manifest.json'))
expected_paths = classical_manifests + quantum_manifests

if not expected_paths:
    raise FileNotFoundError(
        'No se encontraron manifests. Se esperaban archivos *_manifest.json en:'
        f'{CLASSICAL_DIR.as_posix()}\n- {QUANTUM_DIR.as_posix()}'
    )

manifests = load_manifests(expected_paths)
comparison_df = build_comparison_df(manifests)

print('Preview comparativo:')
display(comparison_df.head())

export_comparison(comparison_df, OUT_FILE)
print(f'CSV exportado en: {OUT_FILE.as_posix()}')

Preview comparativo:


,company,model_family,model_name,horizonte,accuracy,f1,train_end_date,created_at_utc
0,BANCO GUAYAQUIL S.A.,classical,BANCO_GUAYAQUIL_SA_h5,h5,0.833333,0.909091,2026-01-21,2026-04-27T17:02:54.735305+00:00
1,BANCO GUAYAQUIL S.A.,quantum,BANCO_GUAYAQUIL_SA_h5,h5,0.366667,0.457143,2026-01-21,2026-04-27T18:12:46.804042+00:00
2,CORPORACION FAVORITA C.A.,classical,CORPORACION_FAVORITA_CA_h5,h5,0.433333,0.514286,2026-01-28,2026-04-27T17:02:55.358077+00:00
3,CORPORACION FAVORITA C.A.,quantum,CORPORACION_FAVORITA_CA_h5,h5,0.400000,0.307692,2026-01-28,2026-04-27T20:49:06.652582+00:00


CSV exportado en: ../results/fase4_comparativa/BVG_subfase8_comparativa_h5.csv
